# Bölüm 10: Dikkat Her Şeydir

> "Dikkat Her Şeydir." — **Vaswani ve arkadaşları**, Google Araştırma, 2017

---

## Neler Öğreneceksiniz

- Statik gömmelerin neden yeterli olmadığı ("river bank" ve "savings bank"deki "bank")
- Dikkat kavramsal düzeyde NE'dir (token'ların birbirine bakması)
- Sorgu, Anahtar, Değer kavramlarından adım adım öz-dikkat nasıl oluşturulur
- Neden dikkat skorlarını ölçeklendirip gelecek token'ları maskeliyoruz
- Tek başlıdan çok başlı dikkate nasıl verimli geçilir
- Dikkati ileri beslemeli ağlarla birleştirerek tam Transformer blokları nasıl oluşturulur
- Dikkat örüntülerinin nasıl görselleştirilir

---

## Kurulum

Önce gerekli paketleri yükleyelim:

In [ ]:
# Gerekli paketleri yükle
!pip install -q torch transformers matplotlib

In [ ]:
# ===== İÇE AKTARMALAR =====
import torch                     # PyTorch: tensör işlemleri
import torch.nn as nn            # Sinir ağı katmanları
import torch.nn.functional as F  # Matematiksel fonksiyonlar (softmax, gelu)
import math                      # Dikkat ölçeklendirmede sqrt için
import matplotlib.pyplot as plt  # Görselleştirme
import numpy as np               # Dizi işlemleri

# Kullanacağımız temel yapı taşları:
# - nn.Linear(in, out): Matris çarpımı katmanı (ağırlıkları öğrenir)
# - F.softmax(x, dim): Skorları olasılıklara çevirir (toplam 1'e)
# - @ operatörü: Matris çarpımı (torch.matmul ile aynı)

## 1. Statik Gömmeler Neden Yeterli Değil

Sorun: Bölüm 9'daki gömmeler **statik** — her token bağlamdan bağımsız olarak aynı vektörü alır.

In [ ]:
# Bölüm 9 GPT2Embeddings'den çıktıyı simüle et
batch_size = 2
seq_len = 6
embed_dim = 768

# Bölüm 9'dan gömmeler (bu örnek için rastgele, ama gerçek olduklarını hayal edin)
embeddings = torch.randn(batch_size, seq_len, embed_dim)
print(f"Girdi gömmeleri şekli: {embeddings.shape}")
# Beklenen çıktı: torch.Size([2, 6, 768])

# Bunlar Bölüm 9'dan STATİK gömmeler
# Amacımız: Bunları BAĞLAM DUYARLI gömmelere dönüştürmek

print("\n'bank' kelimesi her zaman aynı vektörü alır:")
print("- 'river bank' → aynı gömme")
print("- 'savings bank' → aynı gömme")
print("- Ama bunlar tamamen farklı şeyler ifade eder!")

## 2. Adım Adım Öz-Dikkat Oluşturma

Her adımda şekilleri göstererek dikkati kademeli olarak oluşturalım.

**Sezgi:** Her token "Beni anlamak için hangi diğer token'lar ilgili?" diye sorar.
- **Sorgu (Q)**: "Ne arıyorum?"
- **Anahtar (K)**: "Ne sunuyorum?"
- **Değer (V)**: "Paylaşacağım bilgim"

Bir arama motoru gibi düşünün: Sorgu aramanız, Anahtarlar belgelerin başlık/etiketleri, Değerler gerçek içerik.

### Adım 1: Sorgu, Anahtar, Değer Projeksiyonları Oluştur

**`nn.Linear(in_dim, out_dim)` nedir?**
- (in_dim, out_dim) şeklinde bir ağırlık matrisi oluşturur
- Girdiyi geçirdiğinizde: `output = input @ weight`
- Bu ağırlıklar "öğrenilebilir" — eğitim sırasında güncellenir

In [ ]:
# Netlik için daha küçük bir boyut kullanalım
d_model = 768  # Gömmelerden (Bölüm 9)
d_k = 64       # Q, K, V için boyut (tipik: d_model / num_heads)

# Projeksiyon katmanları oluştur (bunların öğrenilebilir parametreleri var!)
W_q = nn.Linear(d_model, d_k, bias=False)  # Sorgu projeksiyonu
W_k = nn.Linear(d_model, d_k, bias=False)  # Anahtar projeksiyonu
W_v = nn.Linear(d_model, d_k, bias=False)  # Değer projeksiyonu

# Gömmeleri Q, K, V'ye projeksle
# Neden Linear? Her rol için en iyi dönüşümü öğrenir
Q = W_q(embeddings)  # (batch, seq, d_k) = (2, 6, 64)
K = W_k(embeddings)  # (batch, seq, d_k) = (2, 6, 64)
V = W_v(embeddings)  # (batch, seq, d_k) = (2, 6, 64)

print(f"Q şekli: {Q.shape}")  # Beklenen: torch.Size([2, 6, 64])
print(f"K şekli: {K.shape}")  # Beklenen: torch.Size([2, 6, 64])
print(f"V şekli: {V.shape}")  # Beklenen: torch.Size([2, 6, 64])

print("\nArtık her token şunlara sahip:")
print("- Q vektörü (64 boyut): 'Ne arıyorum'")
print("- K vektörü (64 boyut): 'Ne sunuyorum'")
print("- V vektörü (64 boyut): 'Paylaşacağım bilgim'")

### Adım 2: Dikkat Skorlarını Hesapla (Q · K^T)

In [ ]:
# Dikkat skorlarını hesapla: Q @ K^T
# K'yı transpoz etmemiz gerekiyor böylece boyutlar matmul için uyumlu olur

# Q şekli: (batch, seq, d_k) = (2, 6, 64)
# K şekli: (batch, seq, d_k) = (2, 6, 64)

# K.transpose(-2, -1) son iki boyutu değiştirir:
# Negatif indeksler: -1 = son boyut, -2 = sondan ikinci
# Yani K (2, 6, 64) → (2, 64, 6) olur

# Matris çarpımı: (2, 6, 64) @ (2, 64, 6) → (2, 6, 6)
scores = Q @ K.transpose(-2, -1)

print(f"Q şekli: {Q.shape}")
print(f"K şekli: {K.shape}")
print(f"K transpoz şekli: {K.transpose(-2, -1).shape}")
print(f"Dikkat skorları şekli: {scores.shape}")
# Beklenen: torch.Size([2, 6, 6])

print(f"\nYığındaki ilk öğe için skorlar:")
print(scores[0])
print("\n[i,j] girdisi = token i'nin token j'ye ne kadar dikkat ettiği")

### Adım 3: Skorları Ölçeklendir

In [ ]:
# sqrt(boyut) ile ölçeklendir
# Neden sqrt? Matematiksel kanıt bunun varyansı stabil tuttuğunu gösterir
scores = scores / math.sqrt(d_k)

print(f"Ölçeklenmiş skorlar şekli: {scores.shape}")  # Hala (2, 6, 6)
print(f"\nÖlçeklendirmeden önce, skor aralığı şu olabilir: ±{d_k}")
print(f"sqrt({d_k}) = {math.sqrt(d_k):.2f} ile ölçeklendirdikten sonra, aralık yaklaşık: ±8")

print("\nBu neden önemli: Ölçeklendirme olmadan, yüksek boyutlu dikkat")
print("neredeyse tüm ağırlığı bir token'a koyar, birden fazla token'a")
print("dikkat etmenin avantajını kaybederiz.")

### Adım 4: Dikkat Ağırlıklarını Almak İçin Softmax Uygula

In [ ]:
# Son boyut üzerinde softmax uygula (anahtarlar boyunca)
# Bu her satırı (her sorguyu) 1'e toplayan olasılıklara çevirir
attn_weights = F.softmax(scores, dim=-1)

print(f"Dikkat ağırlıkları şekli: {attn_weights.shape}")  # Beklenen: (2, 6, 6)
print(f"\nToken 0 için dikkat ağırlıkları (ilk yığın):")
print(attn_weights[0, 0])
# Örnek çıktı: tensor([0.15, 0.20, 0.30, 0.18, 0.10, 0.07])
# Bunlar 1.0'a toplanır!

print(f"\nToken 0 için ağırlıklar toplamı: {attn_weights[0, 0].sum().item():.4f}")
# Beklenen: Token 0 için ağırlıklar toplamı: 1.0000

### Adım 5: Değerlerin Ağırlıklı Toplamı

In [ ]:
# Dikkat ağırlıkları: (batch, seq, seq) = (2, 6, 6)
# Değerler:           (batch, seq, d_k)  = (2, 6, 64)
# İstediğimiz:        (batch, seq, d_k)  = (2, 6, 64)

output = attn_weights @ V

print(f"Çıktı şekli: {output.shape}")  # Beklenen: torch.Size([2, 6, 64])

print(f"\nToken 0 için orijinal gömme (ilk 10 boyut):")
print(embeddings[0, 0, :10])

print(f"\nToken 0 için dikkatten sonra çıktı (ilk 10 boyut):")
print(output[0, 0, :10])
print("\nFarklı değerler! Bu token bağlamı dahil etti")

### Tam Dikkat Fonksiyonu

5 adımı tek bir fonksiyona paketleyelim:

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Ölçeklendirilmiş nokta çarpım dikkati hesapla.
    
    Args:
        Q: Sorgular (batch, seq, d_k)
        K: Anahtarlar (batch, seq, d_k)
        V: Değerler  (batch, seq, d_k)
        mask: Opsiyonel maske (batch, seq, seq)
    
    Returns:
        output: Dikkat çıktısı (batch, seq, d_k)
        attn_weights: Dikkat ağırlıkları (batch, seq, seq)
    """
    d_k = Q.size(-1)  # Sorgu/anahtar boyutunu al
    
    # Adım 1: Q @ K^T skorlarını hesapla
    scores = Q @ K.transpose(-2, -1)  # (batch, seq, seq)
    
    # Adım 2: sqrt(d_k) ile ölçeklendir
    scores = scores / math.sqrt(d_k)
    
    # Adım 3: Sağlanmışsa maske uygula
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Adım 4: Dikkat ağırlıklarını almak için softmax
    attn_weights = F.softmax(scores, dim=-1)  # (batch, seq, seq)
    
    # Adım 5: Değerlerin ağırlıklı toplamı
    output = attn_weights @ V  # (batch, seq, d_k)
    
    return output, attn_weights


# Test et
output, attn_weights = scaled_dot_product_attention(Q, K, V)

print(f"Çıktı şekli: {output.shape}")  # Beklenen: (2, 6, 64)
print(f"Dikkat ağırlıkları şekli: {attn_weights.shape}")  # Beklenen: (2, 6, 6)
print(f"\nİlk token'ın dikkat dağılımı:")
print(attn_weights[0, 0])

## 3. Özbağlanımlı Üretim İçin Nedensel Maskeleme

### Hile Problemi

Maskeleme olmadan, "cat" işlenirken model TÜM token'lara dikkat edebilir — gelecektekiler dahil! Bu eğitim sırasında hile yapmaktır.

In [ ]:
def create_causal_mask(seq_len):
    """
    Nedensel maske oluştur: üst üçgen False (engelle), alt True (izin ver).
    
    Returns:
        mask: (seq_len, seq_len) boolean tensörü
    """
    # torch.tril alt üçgensel matris oluşturur
    # köşegen altında (köşegen dahil) 1'ler, üstte 0'lar
    mask = torch.tril(torch.ones(seq_len, seq_len))
    
    return mask


# seq_len = 6 ile örnek
mask = create_causal_mask(6)
print("Nedensel maske (1 = izin ver, 0 = engelle):")
print(mask)

print("\nMaskeyi okumak:")
print("- Satır 0 (token 0): Yalnızca sütun 0'a dikkat edebilir")
print("- Satır 2 (token 2): Sütun 0, 1, 2'ye dikkat edebilir")
print("- Satır 5 (token 5): Tüm sütunlara 0-5 dikkat edebilir")

### Maskeyi Uygulama

In [ ]:
# Nedensel maske ile test et
Q_test = torch.randn(2, 6, 64)  # (batch, seq, d_k)
K_test = torch.randn(2, 6, 64)
V_test = torch.randn(2, 6, 64)

# Nedensel maske oluştur
causal_mask = create_causal_mask(6)  # (seq, seq)

# Maske ile dikkati uygula
output_masked, attn_weights_masked = scaled_dot_product_attention(
    Q_test, K_test, V_test, mask=causal_mask
)

print("Nedensel maske ile dikkat ağırlıkları (yığındaki ilk öğe):")
print(attn_weights_masked[0])

print("\nDikkat edin: Üst üçgen tamamen sıfır! Gelecek dikkati yok!")

## 4. Çok Başlı Dikkat

### Neden Birden Fazla Baş?

Bir dikkat başı aynı anda yalnızca BİR tür ilişki yakalayabilir. Birden fazla baş modelin farklı örüntüleri aynı anda öğrenmesini sağlar:

| Baş | Ne öğrenebilir |
|------|--------------------|
| Baş 1 | Özne-fiil ilişkileri ("cat" → "sat") |
| Baş 2 | Sıfat-isim bağlantıları ("lazy" → "dog") |
| Baş 3 | Yakındaki kelime örüntüleri (yerel bağlam) |
| Baş 4 | Uzun menzilli bağımlılıklar (zamir çözümlemesi) |

**Temel Fikir:** 64 boyutlu 12 baş = 768 toplam boyut = 1 büyük baş ile aynı!
Ekstra parametre yok — sadece farklı perspektifler.

### Verimli Uygulama

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Verimli çok başlı dikkat (tüm başları birlikte yığınlar).
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model, num_heads'e bölünebilir olmalı"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads  # 768 / 12 = 64
        
        # Birleştirilmiş QKV projeksiyonu (3 kat daha verimli!)
        # Neden 3 * d_model? Çünkü Q, K, V'ye aynı anda projeksiyon yapıyoruz
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        
        # Çıktı projeksiyonu
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        # Dikkat ağırlıklarında dropout
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape
        
        # ===== Adım 1: Q, K, V'ye projeksiyon (hepsini bir kerede!) =====
        qkv = self.qkv_proj(x)  # (batch, seq, 3 * d_model)
        
        # ===== Adım 2: Q, K, V'ye ayır ve çok başlı için yeniden şekillendir =====
        # (batch, seq, 3, num_heads, d_head) şekline dönüştür
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)
        
        # (3, batch, num_heads, seq, d_head) şekline permüte et
        qkv = qkv.permute(2, 0, 3, 1, 4)
        
        # Q, K, V'ye ayır: her biri (batch, num_heads, seq, d_head)
        Q, K, V = qkv[0], qkv[1], qkv[2]
        
        # ===== Adım 3: Ölçeklendirilmiş nokta çarpım dikkati (başlar üzerinde yığınlanmış) =====
        d_k = self.d_head
        scores = Q @ K.transpose(-2, -1)  # (batch, num_heads, seq, seq)
        scores = scores / math.sqrt(d_k)
        
        # Sağlanmışsa nedensel maske uygula
        if mask is not None:
            # Başlar için maskeyi genişlet: (seq, seq) → (1, 1, seq, seq)
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq, seq)
        attn_weights = self.dropout(attn_weights)
        
        # Değerlerin ağırlıklı toplamı
        attn_output = attn_weights @ V  # (batch, num_heads, seq, d_head)
        
        # ===== Adım 4: Başları birleştir =====
        # (batch, seq, num_heads, d_head) şekline transpoz et
        attn_output = attn_output.transpose(1, 2)
        
        # (batch, seq, d_model) şekline yeniden şekillendir — bu başları birleştirir
        attn_output = attn_output.reshape(batch, seq, d_model)
        
        # ===== Adım 5: Son projeksiyon =====
        output = self.out_proj(attn_output)
        
        return output, attn_weights


# Test et
mha = MultiHeadAttention(d_model=768, num_heads=12, dropout=0.1)
embeddings_test = torch.randn(2, 6, 768)
mask_test = create_causal_mask(6)

output_mha, attn_weights_mha = mha(embeddings_test, mask_test)

print(f"Girdi şekli:  {embeddings_test.shape}")     # Beklenen: (2, 6, 768)
print(f"Çıktı şekli: {output_mha.shape}")          # Beklenen: (2, 6, 768)
print(f"Dikkat ağırlıkları şekli: {attn_weights_mha.shape}")  # Beklenen: (2, 12, 6, 6)
print("                                                         ^^ 12 baş!")

## 5. Tam Transformer Blokları

### İleri Beslemeli Ağ

İleri beslemeli ağ boyutu genişletir (768 → 3072), doğrusal olmayan bir fonksiyon uygular, sonra tekrar sıkıştırır (3072 → 768).

**GELU nedir?**
- GELU (Gaussian Error Linear Unit) bir aktivasyon fonksiyonudur
- ReLU gibi ama daha yumuşak — sıfırda sert bir kesim yoktur
- GPT-2, BERT ve çoğu modern transformer'da kullanılır
- Sezgi: Girdi büyüklüğüne bağlı olarak ne kadar sinyalin geçeceğini "kapılar"

In [ ]:
class FeedForward(nn.Module):
    """
    Konuma göre ileri beslemeli ağ.
    Her konuma bağımsız olarak uygulanır (tüm konumlar için aynı ağırlıklar).
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        """
        Args:
            d_model: Model boyutu (GPT-2 small için 768)
            d_ff: İleri beslemeli boyutu (tipik olarak 4 * d_model = 3072)
            dropout: Dropout olasılığı
        """
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (batch, seq, d_model)
        x = self.fc1(x)        # (batch, seq, d_ff) — genişlet
        x = F.gelu(x)          # Doğrusal olmayan
        x = self.dropout(x)
        x = self.fc2(x)        # (batch, seq, d_model) — geri projeksiyon
        return x


# Test et
ffn = FeedForward(d_model=768, d_ff=3072, dropout=0.1)
x_test = torch.randn(2, 6, 768)
output_ffn = ffn(x_test)

print(f"Girdi şekli:  {x_test.shape}")      # Beklenen: (2, 6, 768)
print(f"Çıktı şekli: {output_ffn.shape}")  # Beklenen: (2, 6, 768)

### Tam Transformer Bloğu (Ön-Norm Stili)

**Ön-norm vs Son-norm:**
- **Son-norm** (orijinal Transformer): Her alt katmandan SONRA LayerNorm
- **Ön-norm** (GPT-2, modern): Her alt katmandan ÖNCE LayerNorm

Neden ön-norm? Derin ağlar (12+ katman) için eğitimi daha stabil hale getirir. Gradyanlar artık bağlantılar boyunca daha düzgün akar.

**Artık bağlantılar:** `output = x + sublayer(x)`
- Gradyanların geriye akması için "otoyollar" oluşturur
- Artık bağlantılar olmadan, derin ağlarda gradyanlar kaybolur

In [ ]:
class TransformerBlock(nn.Module):
    """
    Çok başlı dikkat, ileri beslemeli, artık bağlantılar ve 
    katman normalizasyonu ile tam Transformer bloğu (GPT-2 gibi ön-norm stili).
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        """
        Args:
            d_model: Model boyutu (768)
            num_heads: Dikkat başı sayısı (12)
            d_ff: İleri beslemeli boyutu (3072 = 4 * d_model)
            dropout: Dropout olasılığı
        """
        super().__init__()
        
        # Katman normalizasyonu (her alt katmandan önce)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
        # Çok başlı dikkat
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        
        # İleri beslemeli ağ
        self.ffn = FeedForward(d_model, d_ff, dropout)
        
        # Dropout (her alt katmandan sonra uygulanır)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        """
        Args:
            x: Girdi gömmeleri (batch, seq, d_model)
            mask: Nedensel maske (seq, seq) veya (1, 1, seq, seq)
        
        Returns:
            x: Çıktı gömmeleri (batch, seq, d_model)
            attn_weights: Dikkat ağırlıkları (batch, heads, seq, seq)
        """
        # ===== Artık Bağlantı ile Çok Başlı Dikkat =====
        # Ön-norm: Dikkattan ÖNCE normalize et
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)  # Artık bağlantı
        
        # ===== Artık Bağlantı ile İleri Beslemeli =====
        # Ön-norm: İBB'den ÖNCE normalize et
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)  # Artık bağlantı
        
        return x, attn_weights


# Tam bir Transformer bloğunu test et
block = TransformerBlock(
    d_model=768,
    num_heads=12,
    d_ff=3072,
    dropout=0.1
)

# Girdi: Bölüm 9'dan gömmeler
embeddings_block = torch.randn(2, 6, 768)
mask_block = create_causal_mask(6)

# İleri geçiş
output_block, attn_weights_block = block(embeddings_block, mask_block)

print(f"Girdi şekli:  {embeddings_block.shape}")  # Beklenen: (2, 6, 768)
print(f"Çıktı şekli: {output_block.shape}")      # Beklenen: (2, 6, 768)
print(f"Dikkat ağırlıkları şekli: {attn_weights_block.shape}")  # Beklenen: (2, 12, 6, 6)

# Artık bağlantıyı doğrula: çıktı girdiye "benzer" olmalı (tamamen farklı değil)
print(f"\nGirdi ortalaması:  {embeddings_block.mean().item():.4f}")
print(f"Çıktı ortalaması: {output_block.mean().item():.4f}")
print(f"Fark:  {(output_block - embeddings_block).abs().mean().item():.4f}")
print("Fark orta düzeyde olmalı — ne sıfır, ne de çok büyük")

## 6. Dikkat Örüntülerini Görselleştirme

In [ ]:
def visualize_attention(attn_weights, tokens, head_idx=0, ax=None):
    """
    Belirli bir baş için dikkat ağırlıklarını ısı haritası olarak görselleştir.
    
    Args:
        attn_weights: Dikkat ağırlıkları (batch, heads, seq, seq)
        tokens: Token string listesi (uzunluk = seq)
        head_idx: Hangi başın görselleştirileceği
        ax: Matplotlib ekseni (None ise, yeni figür oluştur)
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    # Belirtilen baş için ağırlıkları çıkar (yığındaki ilk öğe)
    weights = attn_weights[0, head_idx].detach().cpu().numpy()
    
    # Isı haritası çiz
    im = ax.imshow(weights, cmap='viridis', aspect='auto', vmin=0, vmax=1)
    
    # İşaretleri ve etiketleri ayarla
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right')
    ax.set_yticklabels(tokens)
    
    # Etiketler
    ax.set_xlabel('Anahtar (dikkat EDİLEN)', fontsize=10)
    ax.set_ylabel('Sorgu (dikkat EDEN)', fontsize=10)
    ax.set_title(f'Dikkat Ağırlıkları - Baş {head_idx}', fontsize=12)
    
    # Renk çubuğu
    plt.colorbar(im, ax=ax, label='Dikkat Ağırlığı')
    
    return ax


# Örnek: Gerçek bir cümleyi işle
from transformers import AutoTokenizer

# Tokenize et
tokenizer = AutoTokenizer.from_pretrained("gpt2")
text = "The quick brown fox jumps over the lazy dog"
token_ids = tokenizer.encode(text, return_tensors="pt")  # (1, seq)

# Token string'lerini al
tokens = [tokenizer.decode([t]) for t in token_ids[0]]
print(f"Token'lar: {tokens}")

# Gömme katmanından geçir (Bölüm 9 çıktısını simüle et)
embed_layer = nn.Embedding(50257, 768)
embeddings_viz = embed_layer(token_ids)  # (1, seq, 768)

# Nedensel maske oluştur
seq_len_viz = embeddings_viz.size(1)
mask_viz = create_causal_mask(seq_len_viz)

# Transformer bloğundan geçir
block_viz = TransformerBlock(d_model=768, num_heads=12, d_ff=3072, dropout=0.1)
output_viz, attn_weights_viz = block_viz(embeddings_viz, mask_viz)

print(f"\nDikkat ağırlıkları şekli: {attn_weights_viz.shape}")  # Beklenen: (1, 12, seq, seq)

# İlk 4 başı görselleştir
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for i, ax in enumerate(axes.flat):
    visualize_attention(attn_weights_viz, tokens, head_idx=i, ax=ax)

plt.tight_layout()
plt.show()

print("\nNe görüyorsunuz:")
print("1. Alt üçgensel örüntü (nedensel maskeleme çalışıyor!)")
print("2. Baş başına farklı örüntüler (çeşitlilik iyidir)")
print("3. Bazı başlar yerel odaklanır, diğerleri uzun menzilli bağımlılıklara")

## 7. Tam Boru Hattı

Ham metinden Transformer bloğu çıktısına kadar veriyi izleyelim:

In [ ]:
# ===== Bölüm 8: Tokenizasyon =====
from transformers import AutoTokenizer
text = "The cat sat on the mat"  # Ham metin
tokenizer = AutoTokenizer.from_pretrained("gpt2")
token_ids = tokenizer.encode(text, return_tensors="pt")  # (1, 6)

print("Adım 1: Tokenizasyon")
print(f"Metin: {text}")
print(f"Token ID'leri: {token_ids}")
print(f"Şekil: {token_ids.shape}\n")

# ===== Bölüm 9: Gömmeler =====
# Bölüm 9'dan GPT2Embeddings'i simüle et
embed_layer = nn.Embedding(50257, 768)
embeddings = embed_layer(token_ids)  # (1, 6, 768)

print("Adım 2: Gömme")
print(f"Gömmeler şekli: {embeddings.shape}")
print(f"İlk token gömmesi (ilk 10 boyut): {embeddings[0, 0, :10]}\n")

# ===== Bölüm 10: Dikkat (BU BÖLÜM) =====
# Nedensel maske oluştur
seq_len_final = token_ids.size(1)  # 6
mask_final = create_causal_mask(seq_len_final)

# Transformer bloğu
block_final = TransformerBlock(d_model=768, num_heads=12, d_ff=3072, dropout=0.1)
output_final, attn_weights_final = block_final(embeddings, mask_final)  # (1, 6, 768)

print("Adım 3: Dikkat (BU BÖLÜM)")
print(f"Çıktı şekli: {output_final.shape}")
print(f"Dikkattan sonra ilk token (ilk 10 boyut): {output_final[0, 0, :10]}\n")

print("Boru hattı tamamlandı!")
print("Ham metin → Token'lar → Gömmeler → Dikkat → Bağlam duyarlı vektörler")

print("\n===== Bölüm 11 Ön İzleme: 12 bloğu istifle =====")
print("Bölüm 11'de çıktıyı 11 Transformer bloğu daha geçireceğiz!")

## Alıştırmalar

### Alıştırma 1: Manuel Dikkat Hesaplaması

Q, K, V matrislerini verilen manuel olarak dikkat skorlarını hesaplayın, softmax uygulayın ve çıktıyı alın. Hesaplamalarınızın `scaled_dot_product_attention()` ile eşleştiğini doğrulayın.

In [ ]:
# Basit 2×3 Q, K, V oluştur (batch=1, seq=2, d_k=3)
Q_ex = torch.tensor([[[1.0, 0.0, 1.0], [0.0, 1.0, 1.0]]])  # (1, 2, 3)
K_ex = torch.tensor([[[1.0, 1.0, 0.0], [0.0, 1.0, 1.0]]])  # (1, 2, 3)
V_ex = torch.tensor([[[2.0, 0.0, 1.0], [1.0, 2.0, 0.0]]])  # (1, 2, 3)

# KODUNUZ BURAYA: 
# 1. scores = Q @ K^T'yi hesapla
# 2. sqrt(d_k) ile ölçeklendir
# 3. Softmax uygula
# 4. V ile ağırlıklı toplam
# 5. scaled_dot_product_attention() ile karşılaştır

### Alıştırma 2: Nedensel Maske Doğrulaması

Nedensel maskeleme ile ve olmadan dikkat ağırlıkları oluşturun. Üst üçgenin maskeleme ile sıfır olduğunu doğrulayın.

In [ ]:
# KODUNUZ BURAYA:
# 1. seq_len=5 için Q, K, V oluştur
# 2. Maske OLMADAN dikkati hesapla
# 3. Nedensel maske İLE dikkati hesapla
# 4. Her iki dikkat ağırlık matrisini yazdır
# 5. Maskeli versiyonda üst üçgenin sıfır olduğunu doğrula

### Alıştırma 3: Çok Başlı Şekiller

Belirli sayılarla `MultiHeadAttention` boyunca şekil dönüşümlerini izleyin.

In [ ]:
# KODUNUZ BURAYA:
# d_model=512, num_heads=8, seq_len=10 ile MHA oluştur
# Her adımdan sonra şekli yazdır:
# 1. Girdi
# 2. QKV projeksiyonundan sonra
# 3. Başları ayırmak için yeniden şekillendirdikten sonra
# 4. Dikkat hesaplamasından sonra
# 5. Başları birleştirdikten sonra
# 6. Çıktı projeksiyonundan sonra

### Alıştırma 4: Tek vs Çok Başlı Karşılaştırma

Aynı girdiyi tek başlı ve 12 başlı dikkatten geçirin. Parametre sayılarını karşılaştırın.

In [ ]:
# KODUNUZ BURAYA:
# 1. Tek başlı dikkat oluştur (d_model=768, num_heads=1)
# 2. Çok başlı dikkat oluştur (d_model=768, num_heads=12)
# 3. Her birinde parametreleri say
# 4. Aynı girdiyi her ikisinden de geçir
# 5. Çıktıları ve parametre sayılarını karşılaştır

### Alıştırma 5: Dikkat Görselleştirmesi

Kendi cümlenizi işleyin ve farklı dikkat başlarını görselleştirin.

In [ ]:
# KODUNUZ BURAYA:
# 1. İlginç bir cümle seçin (örn. "Alice gave Bob a book")
# 2. Tokenize edin
# 3. TransformerBlock'tan geçirin
# 4. Baş 0, 3, 7, 11'i görselleştirin
# 5. Hangi örüntüleri görüyorsunuz (yerel vs uzun menzilli)

### Alıştırma 6: Blokları İstifleme

3 Transformer bloğunu istifleyin ve bir diziyi hepsinden geçirin.

In [ ]:
# KODUNUZ BURAYA:
# 1. 3 ayrı TransformerBlock örneği oluştur
# 2. Gömmeleri blok1 → blok2 → blok3'ten geçir
# 3. Her bloktan sonra şekli yazdır
# 4. Girdi gömmelerini son çıktıyla karşılaştır
# 5. Ne kadar farklılar?

### Alıştırma 7: Ön-Norm vs Son-Norm

LayerNorm'un SONRA geldiği bir son-norm Transformer bloğu uygulayın ve ön-norm ile karşılaştırın.

In [ ]:
# KODUNUZ BURAYA:
# 1. TransformerBlockPostNorm uygulayın
# 2. Aynı girdiyi hem ön-norm hem son-norm'dan geçirin
# 3. Çıktıları karşılaştırın
# 4. Hangisi daha stabil gradyanlara sahip? (gradyan büyüklüklerini kontrol edebilirsiniz)

### Alıştırma 8: Parametre Sayımı

Bir Transformer bloğu için tam parametre sayısını hesaplayın.

In [ ]:
# KODUNUZ BURAYA:
# d_model=768, num_heads=12, d_ff=3072 için:
# 1. QKV projeksiyon parametrelerini say
# 2. Çıktı projeksiyon parametrelerini say
# 3. İBB parametrelerini say (fc1 + fc2)
# 4. LayerNorm parametrelerini say (2 katman)
# 5. Toplamı almak için topla
# 6. model.parameters() ile doğrula

## Bölüm Özeti

**Ne oluşturduk:**

1. Ölçeklendirilmiş nokta çarpım dikkati (Q, K, V → skorlar → softmax → ağırlıklı toplam)
2. Özbağlanımlı üretim için nedensel maskeleme (geleceğe bakmak yok)
3. Verimli çok başlı dikkat (yeniden şekillendirme hilesiyle 1 baş → 12 baş)
4. Tam Transformer blokları (dikkat + İBB + artık + katman normları)
5. Dikkat görselleştirmesi (modelin neye dikkat ettiğini gösteren ısı haritaları)

**Temel kavramlar:**

- **Statik gömmeler** (Bölüm 9) → **Bağlam duyarlı temsiller** (Bölüm 10)
- **Sorgu/Anahtar/Değer**: Farklı roller oynayan üç öğrenilmiş projeksiyon
- **Dikkat ağırlıkları**: İlgililik gösteren softmax olasılıkları
- **Çok başlı**: Farklı başlar farklı örüntüler öğrenir (ekstra parametre yok!)
- **Artık + Normlar**: Derin ağları mümkün kılar (100+ katman)

**Sırada:** Bölüm 11 bu blokları istifleyecek ve tam bir GPT modeli oluşturmak için dil modelleme başlığı ekleyecek!